<a href="https://colab.research.google.com/github/shreyaghora/Bank_Marketing_Project/blob/main/Code/Smote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install lightgbm
!pip install catboost
!pip install xgboost
!pip install specificity
!pip install imbalanced-learn

!pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.0 MB/s eta 0:00:00
time: 328 µs (started: 2026-04-13 16:12:42 +00:00)


In [ ]:
import numpy as np
import pandas as pd
import time
from scipy.stats import uniform


from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix, balanced_accuracy_score

time: 16.8 s (started: 2026-04-13 16:12:52 +00:00)


In [ ]:
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def gmean(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    spec = tn / (tn + fp)
    return np.sqrt(sensitivity * spec)

def fpr_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp / (fp + tn) if (fp + tn) != 0 else 0

time: 3.54 ms (started: 2026-04-13 16:13:08 +00:00)


In [ ]:
df = pd.read_csv("bank-additional-full.csv", sep = ";")
df

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,...,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


time: 264 ms (started: 2026-04-13 16:13:08 +00:00)


In [ ]:
# 3. Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)


# 4. Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])


# 5. Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# 6. One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)
X_shuffled = df.drop('y', axis=1)
y_shuffled = df['y']

# 7. Prepare your data
X_raw = df.drop('y', axis=1)
y_raw = df['y']

# 8. Apply SMOTE
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_raw, y_raw)

# 9. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_train_sm, y_train_sm, test_size=0.2, random_state=42
)


time: 738 ms (started: 2026-04-13 16:13:09 +00:00)


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, solver="liblinear", C = 1.0),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_jobs=-1),
    "Gradient Boosting (GBM)": XGBClassifier(
         random_state=42,
         learning_rate=0.3,
         n_estimators=20
     ),
    "XGBoost": XGBClassifier(
        gamma=0.1,
        reg_lambda=1,
        eval_metric='logloss',
        n_jobs=2
        ),

    "LightGBM": LGBMClassifier(verbose=-1),
    "CatBoost": CatBoostClassifier(verbose=0),
    "Support Vector Machine": LinearSVC(random_state=42, C=1.0,tol=1e-3, max_iter=2000),
    "K Nearest Neighbour": KNeighborsClassifier(),
    "Multi Layer Perceptron": MLPClassifier(
         random_state=42,
         hidden_layer_sizes=(64,32),
         activation='relu',
         solver='adam',
         max_iter=200,
         early_stopping=True
         ),
    "Bagging Classifier": BaggingClassifier(
        estimator=DecisionTreeClassifier(min_samples_split=10, random_state=42),
        bootstrap=True,
        n_jobs=-1),

    "Stacking Classifier": StackingClassifier(
        estimators=[
        ("rf", RandomForestClassifier(n_jobs=-1)),
        ("xgb", XGBClassifier(
            gamma=0.1,
            reg_lambda=1,
            eval_metric='logloss',
            n_jobs=2
            ))
      ],
      final_estimator=LogisticRegression())
}

time: 2.6 ms (started: 2026-04-13 16:13:09 +00:00)


In [ ]:
def evaluate_models(name, X_shuffled, y_shuffled, label):

    result = []
    print(f"\n===== {label} (TUNED) =====")
    print(f"\nTuning Model: {name}\n")
    model = models[name]

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "FPR": fpr,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)

time: 6.4 ms (started: 2026-04-13 16:37:16 +00:00)


In [ ]:
result = []

time: 317 µs (started: 2026-04-13 16:37:19 +00:00)


In [ ]:
model_name = "Logistic Regression"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: Logistic Regression

   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Logistic Regression  0.859781   0.888073  0.824380     0.895416   
1     2  Logistic Regression  0.850068   0.874963  0.813587     0.885877   
2     3  Logistic Regression  0.856498   0.873304  0.832233     0.880512   
3     4  Logistic Regression  0.851847   0.872591  0.825803     0.878163   
4     5  Logistic Regression  0.858413   0.890464  0.819565     0.897796   
5     6  Logistic Regression  0.856908   0.879791  0.827417     0.886513   
6     7  Logistic Regression  0.855110   0.894474  0.812984     0.899327   
7     8  Logistic Regression  0.854563   0.876363  0.828587     0.881005   
8     9  Logistic Regression  0.860446   0.877723  0.834576     0.885846   
9    10  Logistic Regression  0.849501   0.870812  0.811077     0.885928   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.855042  0.859164  0.104584  0.

In [ ]:
model_name = "Decision Tree"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: Decision Tree

   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Decision Tree  0.896990   0.888741  0.908372     0.885534  0.898449   
1     2  Decision Tree  0.899042   0.884913  0.915217     0.883166  0.899810   
2     3  Decision Tree  0.904378   0.892542  0.918317     0.890582  0.905246   
3     4  Decision Tree  0.900547   0.885834  0.920795     0.880088  0.902976   
4     5  Decision Tree  0.907798   0.904685  0.913043     0.902479  0.908845   
5     6  Decision Tree  0.902326   0.890154  0.918351     0.886239  0.904032   
6     7  Decision Tree  0.904501   0.900131  0.915041     0.893438  0.907525   
7     8  Decision Tree  0.903954   0.892660  0.920260     0.887355  0.906250   
8     9  Decision Tree  0.905185   0.891444  0.920740     0.889913  0.905855   
9    10  Decision Tree  0.902449   0.888950  0.913691     0.891791  0.901151   

         GM       FPR       AUC       MCC     Kappa  Balanced 

In [ ]:
model_name = "Random Forest"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: Random Forest

   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Random Forest  0.934610   0.938411  0.930734     0.938512  0.934556   
1     2  Random Forest  0.931327   0.932114  0.929025     0.933586  0.930567   
2     3  Random Forest  0.941860   0.938302  0.945270     0.938487  0.941773   
3     4  Random Forest  0.937073   0.930831  0.945019     0.929043  0.937871   
4     5  Random Forest  0.938167   0.943407  0.933152     0.943251  0.938251   
5     6  Random Forest  0.937209   0.935545  0.939377     0.935033  0.937457   
6     7  Random Forest  0.938569   0.945376  0.934010     0.943354  0.939659   
7     8  Random Forest  0.933096   0.931695  0.935991     0.930149  0.933838   
8     9  Random Forest  0.939663   0.936334  0.942281     0.937093  0.939298   
9    10  Random Forest  0.939937   0.938661  0.937869     0.941898  0.938265   

         GM       FPR       AUC       MCC     Kappa  Balanced 

In [ ]:
model_name = "Gradient Boosting (GBM)"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: Gradient Boosting (GBM)

   Fold               Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Gradient Boosting (GBM)  0.879891   0.920410  0.832561     0.927532   
1     2  Gradient Boosting (GBM)  0.878933   0.911800  0.836509     0.920575   
2     3  Gradient Boosting (GBM)  0.887141   0.919928  0.846810     0.927055   
3     4  Gradient Boosting (GBM)  0.884405   0.910357  0.854110     0.915017   
4     5  Gradient Boosting (GBM)  0.884542   0.924043  0.839674     0.930028   
5     6  Gradient Boosting (GBM)  0.887004   0.916814  0.851720     0.922423   
6     7  Gradient Boosting (GBM)  0.883431   0.928805  0.836495     0.932698   
7     8  Gradient Boosting (GBM)  0.879464   0.915581  0.838351     0.921314   
8     9  Gradient Boosting (GBM)  0.889315   0.915485  0.855565     0.922451   
9    10  Gradient Boosting (GBM)  0.876727   0.912935  0.825415     0.925373   

         F1        GM       FPR       AUC       MCC 

In [ ]:
model_name = "XGBoost"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: XGBoost

   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1    XGBoost  0.912312   0.942657  0.878647     0.946198  0.909527   
1     2    XGBoost  0.904514   0.935359  0.867164     0.941176  0.899971   
2     3    XGBoost  0.917784   0.941776  0.889714     0.945563  0.915005   
3     4    XGBoost  0.916416   0.936698  0.894121     0.938944  0.914914   
4     5    XGBoost  0.913817   0.948529  0.876359     0.951791  0.911017   
5     6    XGBoost  0.915458   0.939630  0.888312     0.942708  0.913251   
6     7    XGBoost  0.912300   0.949826  0.874967     0.951486  0.910861   
7     8    XGBoost  0.908606   0.937409  0.877407     0.940364  0.906416   
8     9    XGBoost  0.916952   0.938336  0.890914     0.942516  0.914010   
9    10    XGBoost  0.906143   0.935132  0.867304     0.942964  0.899942   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.911797  0.053802  0.912423  0.826

In [ ]:
model_name = "LightGBM"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: LightGBM

   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1   LightGBM  0.899179   0.929619  0.864467     0.934120  0.895860   
1     2   LightGBM  0.895759   0.926335  0.857774     0.933044  0.890737   
2     3   LightGBM  0.907524   0.932243  0.877888     0.936854  0.904249   
3     4   LightGBM  0.900821   0.921888  0.876973     0.924917  0.898870   
4     5   LightGBM  0.901915   0.934840  0.865489     0.938843  0.898829   
5     6   LightGBM  0.905882   0.928530  0.879847     0.932018  0.903533   
6     7   LightGBM  0.903407   0.941040  0.865616     0.943073  0.901753   
7     8   LightGBM  0.896976   0.927697  0.863032     0.931530  0.894197   
8     9   LightGBM  0.908059   0.930260  0.880420     0.935195  0.904654   
9    10   LightGBM  0.897934   0.925522  0.859432     0.934435  0.891254   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.898619  0.065880  0.899294  0.80

In [ ]:
model_name = "CatBoost"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: CatBoost

   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1   CatBoost  0.923393   0.945512  0.899100     0.947845  0.921722   
1     2   CatBoost  0.918057   0.941555  0.889809     0.945785  0.914951   
2     3   CatBoost  0.929822   0.948836  0.907866     0.951551  0.927899   
3     4   CatBoost  0.924624   0.940231  0.907730     0.941694  0.923695   
4     5   CatBoost  0.924761   0.952312  0.895380     0.954545  0.922969   
5     6   CatBoost  0.925581   0.943150  0.906062     0.945175  0.924234   
6     7   CatBoost  0.925571   0.954274  0.897676     0.954851  0.925110   
7     8   CatBoost  0.922151   0.945683  0.897206     0.947543  0.920807   
8     9   CatBoost  0.925708   0.942750  0.904999     0.946041  0.923489   
9    10   CatBoost  0.918457   0.938927  0.890357     0.945096  0.913997   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.923151  0.052155  0.923473  0.84

In [ ]:
model_name = "Support Vector Machine"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: Support Vector Machine

   Fold              Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Support Vector Machine  0.858550   0.889152  0.820289     0.897063   
1     2  Support Vector Machine  0.849111   0.878533  0.806959     0.890485   
2     3  Support Vector Machine  0.838440   0.867185  0.797305     0.879151   
3     4  Support Vector Machine  0.850889   0.874059  0.821720     0.880363   
4     5  Support Vector Machine  0.856635   0.890736  0.815217     0.898623   
5     6  Support Vector Machine  0.851710   0.880236  0.814855     0.888706   
6     7  Support Vector Machine  0.854152   0.898482  0.806305     0.904375   
7     8  Support Vector Machine  0.852921   0.879431  0.820993     0.885422   
8     9  Support Vector Machine  0.860856   0.881594  0.830710     0.890456   
9    10  Support Vector Machine  0.848817   0.873325  0.806297     0.889126   

         F1        GM       FPR       AUC       MCC     Kappa  \

In [ ]:
model_name = "K Nearest Neighbour"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: K Nearest Neighbour

   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  K Nearest Neighbour  0.910397   0.875000  0.958277     0.862201   
1     2  K Nearest Neighbour  0.901368   0.861776  0.953880     0.849824   
2     3  K Nearest Neighbour  0.913680   0.878750  0.958746     0.869080   
3     4  K Nearest Neighbour  0.908345   0.872335  0.957812     0.858361   
4     5  K Nearest Neighbour  0.915321   0.881006  0.961685     0.868320   
5     6  K Nearest Neighbour  0.910944   0.878361  0.954397     0.867325   
6     7  K Nearest Neighbour  0.913805   0.884606  0.956452     0.869041   
7     8  K Nearest Neighbour  0.910521   0.878274  0.954977     0.865268   
8     9  K Nearest Neighbour  0.917499   0.880484  0.964374     0.871475   
9    10  K Nearest Neighbour  0.910795   0.873106  0.955581     0.868337   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.914747  0.908971  0.137799  0.

In [ ]:
model_name = "Multi Layer Perceptron"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: Multi Layer Perceptron

   Fold              Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Multi Layer Perceptron  0.844460   0.866879  0.815108     0.874005   
1     2  Multi Layer Perceptron  0.746512   0.748594  0.735156     0.757658   
2     3  Multi Layer Perceptron  0.852394   0.870687  0.825908     0.878606   
3     4  Multi Layer Perceptron  0.825992   0.846709  0.798312     0.853960   
4     5  Multi Layer Perceptron  0.773187   0.794237  0.741576     0.805234   
5     6  Multi Layer Perceptron  0.856635   0.864067  0.847078     0.866228   
6     7  Multi Layer Perceptron  0.818853   0.852727  0.781192     0.858385   
7     8  Multi Layer Perceptron  0.819674   0.889766  0.733388     0.907510   
8     9  Multi Layer Perceptron  0.850458   0.842362  0.858879     0.842191   
9    10  Multi Layer Perceptron  0.840060   0.848511  0.817262     0.861674   

         F1        GM       FPR       AUC       MCC     Kappa  \

In [ ]:
model_name = "Bagging Classifier"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: Bagging Classifier

   Fold          Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Bagging Classifier  0.927907   0.934662  0.920644     0.935218   
1     2  Bagging Classifier  0.916279   0.920369  0.909693     0.922743   
2     3  Bagging Classifier  0.929001   0.929931  0.927118     0.930866   
3     4  Bagging Classifier  0.923803   0.921330  0.927599     0.919967   
4     5  Bagging Classifier  0.925445   0.936508  0.913859     0.937190   
5     6  Bagging Classifier  0.928317   0.930099  0.926543     0.930099   
6     7  Bagging Classifier  0.928171   0.939138  0.919316     0.937465   
7     8  Bagging Classifier  0.925571   0.931849  0.919718     0.931530   
8     9  Bagging Classifier  0.928581   0.924637  0.931787     0.925434   
9    10  Bagging Classifier  0.925434   0.929549  0.916222     0.934168   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.927600  0.927902  0.064782  0.927931  0.85

In [ ]:
model_name = "Stacking Classifier"
curr_res = evaluate_models(model_name, X_train_sm, y_train_sm, "SMOTE")
result.append(curr_res)
print(curr_res)


===== SMOTE (TUNED) =====

Tuning Model: Stacking Classifier

   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Stacking Classifier  0.856498   0.958655  0.746114     0.967609   
1     2  Stacking Classifier  0.858276   0.953032  0.750898     0.963676   
2     3  Stacking Classifier  0.848153   0.937630  0.744224     0.951007   
3     4  Stacking Classifier  0.846375   0.941808  0.740065     0.953795   
4     5  Stacking Classifier  0.838988   0.942695  0.724185     0.955372   
5     6  Stacking Classifier  0.846101   0.944620  0.735937     0.956689   
6     7  Stacking Classifier  0.846901   0.952726  0.737644     0.961582   
7     8  Stacking Classifier  0.840881   0.933677  0.736913     0.946715   
8     9  Stacking Classifier  0.864277   0.954059  0.762773     0.963937   
9    10  Stacking Classifier  0.861130   0.954253  0.750633     0.965885   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.839135  0.849674  0.032391  0.

In [ ]:
result

[   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
 0     1  Logistic Regression  0.859781   0.888073  0.824380     0.895416   
 1     2  Logistic Regression  0.850068   0.874963  0.813587     0.885877   
 2     3  Logistic Regression  0.856498   0.873304  0.832233     0.880512   
 3     4  Logistic Regression  0.851847   0.872591  0.825803     0.878163   
 4     5  Logistic Regression  0.858413   0.890464  0.819565     0.897796   
 5     6  Logistic Regression  0.856908   0.879791  0.827417     0.886513   
 6     7  Logistic Regression  0.855110   0.894474  0.812984     0.899327   
 7     8  Logistic Regression  0.854563   0.876363  0.828587     0.881005   
 8     9  Logistic Regression  0.860446   0.877723  0.834576     0.885846   
 9    10  Logistic Regression  0.849501   0.870812  0.811077     0.885928   
 
          F1        GM       FPR       AUC       MCC     Kappa  \
 0  0.855042  0.859164  0.104584  0.859898  0.721495  0.719625   
 1  0.843160  0.848

time: 61.3 ms (started: 2026-04-13 16:52:46 +00:00)


In [ ]:
separator = pd.DataFrame([{
        "Fold": "",
        "Classifier": "",
        "Accuracy": "",
        "Precision": "",
        "Recall": "",
        "Specificity": "",
        "F1": "",
        "GM": "",
        "FPR": "",
        "AUC": "",
        "MCC": "",
        "Kappa": "",
        "Balanced Accuracy": "",
        "Training Time (s)": ""
    }])
result_df = result[0]
# with pd.ExcelWriter("smote.xlsx") as writer:
for i in range(1,12):
    result_df = pd.concat([result_df, separator, result[i]], ignore_index=True)
    #  results_df.to_excel(writer, separator, index=False)
result_df.to_excel("result.xlsx")

time: 528 ms (started: 2026-04-13 16:55:03 +00:00)
